# v7_overnight_cpu — resumable overnight NLV experiments

This notebook queues the two remaining **high-value CPU experiments**:

1. **`capped15_endpoint`** — MSE, exactly 15 daily SDE updates, no patience/monitor-based model selection.
2. **`capped20_twobatch`** — exactly 20 daily SDE updates using the Sabaté/Gierjatowicz-inspired **two-independent-batch MSE gradient estimator**.

Both use the already-fixed v7 architecture, 4096 paths per batch, 400 contracts per optimizer step, 10 SDE optimizer steps per update, the alternating learned control-variate/hedge phase, 3 PyTorch CPU threads, and 100k common-shock next-day evaluation.

### Overnight safety and visibility
- **No Huber waiting or Huber work is included**; that branch is now finished.
- Each branch has a **rolling-date progress bar**, and each training date has a nested **SDE-update progress bar** with live train/hedge losses.
- Each new branch saves a **full mid-date resume checkpoint every 2 SDE updates**.
- Rerun the notebook after an interruption: completed dates are reused and an interrupted date resumes from its latest 2-update checkpoint.
- It reuses the existing `v7_long_run_workspace/` from `v7_long_runs.ipynb` and **does not overwrite** current progress.

The fixed-endpoint design is intentional: after the recent diagnostics, same-day monitor/rerank selection is *not* used to choose the daily model in these two experiments.


In [1]:
from pathlib import Path
import os, sys, zipfile, subprocess, time, json, shutil

ROOT = Path.cwd()
WORK = ROOT / "v7_long_run_workspace"

def find_local(filename):
    exact = ROOT / filename
    if exact.exists():
        return exact
    stem = Path(filename).stem
    suffix = Path(filename).suffix
    candidates = sorted(ROOT.glob(stem + "*" + suffix))
    if candidates:
        return candidates[0]
    return None

SCRIPTS_ZIP = find_local("neural_lv_v7_experiment_scripts_recovery.zip")
CACHE_ZIP = find_local("neural_lv_v7_optimization_cache_results.zip")
LONG_RESULTS_ZIP = find_local("v7_long_runs_results.zip")

print("ROOT:", ROOT)
print("WORK:", WORK)
print("scripts:", SCRIPTS_ZIP)
print("cache:", CACHE_ZIP)
print("prior long-run results:", LONG_RESULTS_ZIP)

# Preserve an existing workspace (especially the Huber process/results).
if not (WORK / "search_pipeline.py").exists():
    if SCRIPTS_ZIP is None or CACHE_ZIP is None:
        raise FileNotFoundError(
            "No existing v7_long_run_workspace was found. Put the two recovery ZIPs "
            "beside this notebook and rerun."
        )
    print("Initialising workspace from recovery bundles...")
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SCRIPTS_ZIP, "r") as z:
        z.extractall(WORK)
    with zipfile.ZipFile(CACHE_ZIP, "r") as z:
        z.extractall(WORK)

    # If available, merge the completed robust_mse / partial Huber work created
    # by the earlier local notebook. This happens only on a fresh workspace.
    if LONG_RESULTS_ZIP is not None:
        with zipfile.ZipFile(LONG_RESULTS_ZIP, "r") as z:
            z.extractall(WORK)
else:
    print("Existing workspace detected; no recovery archive will be re-extracted.")

(WORK / ".v7_overnight_cpu_initialized").write_text(
    "preserve workspace\n", encoding="utf-8"
)

# audit_v7.py in the recovery archive contains the old chat-sandbox notebook path.
# Patch it to the copy stored directly in this persistent workspace.
audit_path = WORK / "audit_v7.py"
audit_text = audit_path.read_text(encoding="utf-8")
import re
audit_text = re.sub(
    r'^NOTEBOOK\s*=.*$',
    'NOTEBOOK = ROOT / "05_Neural_LV_v7_final_revision_crash_safe.ipynb"',
    audit_text,
    count=1,
    flags=re.MULTILINE,
)
audit_path.write_text(audit_text, encoding="utf-8")

print("Workspace ready.")


ROOT: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work
WORK: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\v7_long_run_workspace
scripts: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\neural_lv_v7_experiment_scripts_recovery.zip
cache: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\neural_lv_v7_optimization_cache_results.zip
prior long-run results: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\v7_long_runs_results.zip
Existing workspace detected; no recovery archive will be re-extracted.
Workspace ready.


In [2]:
RUNNER = WORK / "overnight_fixed_runner.py"
RUNNER.write_text('\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom tqdm import tqdm\n\nimport search_pipeline as sp\nfrom audit_v7 import load_notebook_environment\n\n\nSAVE_EVERY_UPDATES = 2\nNUM_THREADS = 3\n\nPRESETS = {\n    "capped15_endpoint": {\n        "objective": "mse",\n        "updates": 15,\n        "paths_per_step": 4096,\n        "seed_offset": 0,\n    },\n    "capped20_twobatch": {\n        "objective": "two_batch_mse",\n        "updates": 20,\n        "paths_per_step": 4096,\n        # Separate training stream from capped MSE while keeping evaluation\n        # randomness common across variants.\n        "seed_offset": 300_000,\n    },\n}\n\n\ndef atomic_torch_save(payload, path: Path):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(payload, tmp)\n    os.replace(tmp, path)\n\n\ndef cpu_optimizer_state(state):\n    """Move tensor-valued Adam state to CPU for a portable resume file."""\n    out = {}\n    for key, value in state.items():\n        if isinstance(value, dict):\n            out[key] = cpu_optimizer_state(value)\n        elif isinstance(value, list):\n            out[key] = [\n                item.detach().cpu().clone() if torch.is_tensor(item) else item\n                for item in value\n            ]\n        elif torch.is_tensor(value):\n            out[key] = value.detach().cpu().clone()\n        else:\n            out[key] = value\n    return out\n\n\ndef load_finished_model(ns, checkpoint_path: Path):\n    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    model = ns["MaturityHeadNeuralLocalVol"](payload["boundaries"]).cpu()\n    model.load_state_dict(payload["final_model_state"], strict=True)\n    model.eval()\n    sp.set_grad(model, False)\n    return model\n\n\ndef existing_row(path: Path, date: pd.Timestamp):\n    if not path.exists():\n        return None\n    frame = pd.read_csv(path)\n    if "date" not in frame.columns:\n        return None\n    frame["date"] = pd.to_datetime(frame["date"])\n    hit = frame.loc[frame["date"] == pd.Timestamp(date)]\n    return None if hit.empty else hit.iloc[-1].to_dict()\n\n\ndef save_middate(\n    path, signature, date, previous_date, model, hedge,\n    opt_model, opt_hedge, sampler, history, update, elapsed\n):\n    payload = {\n        "version": "v7_overnight_fixed_middate_v1",\n        "signature": signature,\n        "date": str(pd.Timestamp(date).date()),\n        "previous_date": str(pd.Timestamp(previous_date).date()),\n        "model_state": sp.cpu_state_dict(model),\n        "hedge_state": {\n            k: v.detach().cpu().clone() for k, v in hedge.state_dict().items()\n        },\n        "opt_model": cpu_optimizer_state(opt_model.state_dict()),\n        "opt_hedge": cpu_optimizer_state(opt_hedge.state_dict()),\n        "sampler_rng_state": sampler.rng.bit_generator.state,\n        "torch_rng_state": torch.get_rng_state(),\n        "history": history,\n        "last_update": int(update),\n        "elapsed_seconds": float(elapsed),\n    }\n    atomic_torch_save(payload, path)\n\n\ndef train_fixed_endpoint(\n    ns, *, name, objective, updates, paths_per_step, seed_offset,\n    date, previous_model, previous_date, output_dir\n):\n    cfg = sp.SearchConfig(\n        name=name,\n        objective=objective,\n        paths_per_step=paths_per_step,\n        steps_per_phase=10,\n        contracts_per_step=400,\n        max_sde_updates=updates,\n        early_min_sde=updates + 1,          # fixed endpoint: no early stopping\n        early_patience_sde=updates + 1,\n        monitor_paths=0,                    # monitor is intentionally not a selector\n        sde_lr=1e-4,\n        hedge_lr=1e-3,\n        eval_paths=100000,\n        seed_offset=seed_offset,\n    )\n    signature = {\n        "name": name,\n        "objective": objective,\n        "updates": updates,\n        "paths_per_step": paths_per_step,\n        "steps_per_phase": cfg.steps_per_phase,\n        "contracts_per_step": cfg.contracts_per_step,\n        "sde_lr": cfg.sde_lr,\n        "hedge_lr": cfg.hedge_lr,\n        "seed_offset": seed_offset,\n    }\n\n    selected, _, _ = ns["daily_split"](date)\n    ctx = ns["build_calibration_context"](date, selected)\n\n    # Preserve the construction/order of the recovered search implementation.\n    model = ns["MaturityHeadNeuralLocalVol"](ctx.maturity_boundaries).cpu()\n    ns["transfer_rolling_model"](previous_model, model, previous_date, date)\n    hedge = ns["IntervalHedgeBank"](ctx.maturity_boundaries, len(ctx.contracts)).cpu()\n\n    opt_model = torch.optim.Adam(model.parameters(), lr=cfg.sde_lr)\n    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=cfg.hedge_lr)\n    sampler = ns["UniformContractSampler"](\n        ctx,\n        cfg.contracts_per_step,\n        ns["SEED"] + seed_offset + int(pd.Timestamp(date).day) * 101,\n    )\n\n    stamp = f"{pd.Timestamp(date):%Y%m%d}"\n    progress_path = output_dir / f"{stamp}_inprogress.pt"\n    history = []\n    start_update = 0\n    elapsed_before = 0.0\n\n    if progress_path.exists():\n        p = torch.load(progress_path, map_location="cpu", weights_only=False)\n        if p.get("signature") != signature:\n            raise RuntimeError(\n                f"Resume signature mismatch for {progress_path}.\\n"\n                f"saved={p.get(\'signature\')}\\ncurrent={signature}"\n            )\n        model.load_state_dict(p["model_state"], strict=True)\n        hedge.load_state_dict(p["hedge_state"], strict=True)\n        opt_model.load_state_dict(p["opt_model"])\n        opt_hedge.load_state_dict(p["opt_hedge"])\n        sampler.rng.bit_generator.state = p["sampler_rng_state"]\n        torch.set_rng_state(p["torch_rng_state"])\n        history = list(p.get("history", []))\n        start_update = int(p["last_update"])\n        elapsed_before = float(p.get("elapsed_seconds", 0.0))\n        print(\n            f"[resume] {name} {pd.Timestamp(date).date()} "\n            f"from update {start_update}/{updates}",\n            flush=True,\n        )\n        del p\n        gc.collect()\n    else:\n        # Same date-based training seed convention as the recovered search.\n        torch.manual_seed(\n            ns["SEED"] + seed_offset + int(pd.Timestamp(date).day) * 17\n        )\n\n    wall_start = time.time()\n\n    update_bar = tqdm(\n        range(start_update + 1, updates + 1),\n        total=updates,\n        initial=start_update,\n        desc=f"{name} {pd.Timestamp(date):%m-%d} SDE updates",\n        unit="upd",\n        leave=False,\n        dynamic_ncols=True,\n    )\n    for update in update_bar:\n        # ----- SDE/model phase -----\n        model.train()\n        hedge.eval()\n        sp.set_grad(model, True)\n        sp.set_grad(hedge, False)\n        model_losses = []\n\n        for _ in range(cfg.steps_per_phase):\n            ids = sampler.draw()\n            opt_model.zero_grad(set_to_none=True)\n            loss, scratch = sp.training_loss(\n                ns, model, hedge, ctx, ids, objective, cfg\n            )\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            opt_model.step()\n            model_losses.append(float(loss.detach()))\n            del loss, scratch\n\n        # ----- Sabate-style learned control-variate / hedge phase -----\n        model.eval()\n        hedge.train()\n        sp.set_grad(model, False)\n        sp.set_grad(hedge, True)\n        hedge_losses = []\n\n        for _ in range(cfg.steps_per_phase):\n            ids = sampler.draw()\n            z = torch.randn(\n                cfg.paths_per_step,\n                len(ctx.grid_np) - 1,\n                dtype=torch.float32,\n            )\n            opt_hedge.zero_grad(set_to_none=True)\n            _, cv = ns["simulate_estimators"](\n                model, hedge, ctx, ids, z, "hedge"\n            )\n            variance = torch.var(\n                cv / ctx.spot, dim=0, unbiased=False\n            ).mean()\n            variance.backward()\n            nn.utils.clip_grad_norm_(hedge.parameters(), 3.0)\n            opt_hedge.step()\n            hedge_losses.append(float(variance.detach()))\n            del z, cv, variance\n\n        history.append({\n            "update": update,\n            "train_loss": float(np.mean(model_losses)),\n            "hedge_variance_loss": float(np.mean(hedge_losses)),\n        })\n        update_bar.set_postfix(\n            train=f"{history[-1][\'train_loss\']:.3g}",\n            hedge=f"{history[-1][\'hedge_variance_loss\']:.3g}",\n            refresh=False,\n        )\n\n        elapsed = elapsed_before + (time.time() - wall_start)\n        if (\n            update % SAVE_EVERY_UPDATES == 0\n            or update == updates\n        ):\n            save_middate(\n                progress_path, signature, date, previous_date,\n                model, hedge, opt_model, opt_hedge, sampler,\n                history, update, elapsed,\n            )\n            print(\n                f"[checkpoint] {name} {pd.Timestamp(date).date()} "\n                f"{update}/{updates} | train={history[-1][\'train_loss\']:.6g}",\n                flush=True,\n            )\n        gc.collect()\n\n    model.eval()\n    sp.set_grad(model, False)\n\n    history_frame = pd.DataFrame(history)\n    history_frame.to_csv(\n        output_dir / f"{stamp}_history.csv", index=False\n    )\n\n    checkpoint = {\n        "version": "v7_overnight_fixed_final_v1",\n        "config": signature,\n        "date": str(pd.Timestamp(date).date()),\n        "previous_date": str(pd.Timestamp(previous_date).date()),\n        "boundaries": ctx.maturity_boundaries,\n        "endpoint_update": int(updates),\n        "final_model_state": sp.cpu_state_dict(model),\n    }\n    atomic_torch_save(checkpoint, output_dir / f"{stamp}_checkpoint.pt")\n    progress_path.unlink(missing_ok=True)\n\n    meta = {\n        "date": pd.Timestamp(date),\n        "endpoint_update": int(updates),\n        "objective": objective,\n        "paths_per_step": int(paths_per_step),\n        "elapsed_seconds": elapsed_before + (time.time() - wall_start),\n    }\n\n    del hedge, opt_model, opt_hedge\n    gc.collect()\n    return model, ctx, meta\n\n\ndef weighted_pooled(frame: pd.DataFrame):\n    n = frame["otm_n"].to_numpy(float)\n    pricing = frame["otm_pricing_RMSE"].to_numpy(float)\n    hedge = frame["otm_NLV_hedge_RMSE"].to_numpy(float)\n    market = frame["otm_market_delta_hedge_RMSE"].to_numpy(float)\n    return {\n        "otm_n": int(n.sum()),\n        "pooled_otm_pricing_RMSE": float(\n            np.sqrt(np.sum(n * pricing**2) / np.sum(n))\n        ),\n        "pooled_otm_NLV_hedge_RMSE": float(\n            np.sqrt(np.sum(n * hedge**2) / np.sum(n))\n        ),\n        "pooled_otm_market_delta_hedge_RMSE": float(\n            np.sqrt(np.sum(n * market**2) / np.sum(n))\n        ),\n        "weighted_otm_pricing_MAE": float(\n            np.sum(n * frame["otm_pricing_MAE"].to_numpy(float)) / np.sum(n)\n        ),\n    }\n\n\ndef run_variant(preset_name: str):\n    if preset_name not in PRESETS:\n        raise KeyError(preset_name)\n    preset = PRESETS[preset_name]\n\n    output_dir = sp.RUN_ROOT / preset_name\n    output_dir.mkdir(parents=True, exist_ok=True)\n    (output_dir / "config.json").write_text(\n        json.dumps(preset, indent=2), encoding="utf-8"\n    )\n\n    ns = load_notebook_environment()\n    ns["stage_log"] = lambda message: None\n    ns["FINAL_PATH_CHUNK"] = 2000\n    torch.set_num_threads(NUM_THREADS)\n\n    previous = sp.load_anchor(ns)\n    previous_date = pd.Timestamp("2024-03-01")\n\n    dates = [\n        pd.Timestamp("2024-03-04"),\n        pd.Timestamp("2024-03-05"),\n        pd.Timestamp("2024-03-06"),\n        pd.Timestamp("2024-03-07"),\n    ]\n    next_dates = [\n        pd.Timestamp("2024-03-05"),\n        pd.Timestamp("2024-03-06"),\n        pd.Timestamp("2024-03-07"),\n        pd.Timestamp("2024-03-08"),\n    ]\n\n    partial_summary_path = output_dir / "development_summary_partial.csv"\n    partial_meta_path = output_dir / "training_metadata_partial.csv"\n    summaries = []\n    metadata = []\n\n    date_pairs = list(zip(dates, next_dates))\n    date_bar = tqdm(\n        date_pairs,\n        total=len(date_pairs),\n        desc=f"{preset_name} rolling dates",\n        unit="date",\n        dynamic_ncols=True,\n    )\n    for index, (date, next_date) in enumerate(date_bar, start=1):\n        date_bar.set_postfix_str(f"{date:%m-%d}->{next_date:%m-%d}", refresh=False)\n        checkpoint_path = output_dir / f"{date:%Y%m%d}_checkpoint.pt"\n\n        existing_summary = existing_row(partial_summary_path, date)\n        existing_meta = existing_row(partial_meta_path, date)\n\n        if checkpoint_path.exists():\n            print(\n                f"[reuse endpoint] {preset_name} {date.date()}",\n                flush=True,\n            )\n            model = load_finished_model(ns, checkpoint_path)\n            ctx = None\n            if existing_meta is None:\n                payload = torch.load(\n                    checkpoint_path, map_location="cpu", weights_only=False\n                )\n                meta = {\n                    "date": date,\n                    "endpoint_update": int(payload["endpoint_update"]),\n                    "objective": preset["objective"],\n                    "paths_per_step": int(preset["paths_per_step"]),\n                    "elapsed_seconds": np.nan,\n                }\n                del payload\n            else:\n                meta = existing_meta\n        else:\n            print(\n                f"[{preset_name}] train {date.date()} "\n                f"({preset[\'updates\']} fixed updates)",\n                flush=True,\n            )\n            model, ctx, meta = train_fixed_endpoint(\n                ns,\n                name=preset_name,\n                objective=preset["objective"],\n                updates=int(preset["updates"]),\n                paths_per_step=int(preset["paths_per_step"]),\n                seed_offset=int(preset["seed_offset"]),\n                date=date,\n                previous_model=previous,\n                previous_date=previous_date,\n                output_dir=output_dir,\n            )\n\n        if existing_summary is None:\n            print(\n                f"[{preset_name}] evaluate {date.date()} -> "\n                f"{next_date.date()} at 100k common-shock paths",\n                flush=True,\n            )\n            summary = sp.evaluate_one_transition(\n                ns,\n                model,\n                date,\n                next_date,\n                sp.SearchConfig(\n                    name=preset_name,\n                    eval_paths=100000,\n                    seed_offset=int(preset["seed_offset"]),\n                ),\n                output_dir,\n                index,\n            )\n        else:\n            summary = existing_summary\n\n        summaries.append(summary)\n        metadata.append(meta)\n\n        pd.DataFrame(summaries).to_csv(\n            partial_summary_path, index=False\n        )\n        pd.DataFrame(metadata).to_csv(\n            partial_meta_path, index=False\n        )\n\n        if ctx is not None:\n            del ctx\n        del previous\n        previous = model\n        previous_date = date\n        gc.collect()\n\n    date_bar.close()\n\n    first = pd.read_csv(\n        sp.BASE_CACHE / "v7_first_transition_summary.csv"\n    ).iloc[0].to_dict()\n    combined = pd.concat(\n        [pd.DataFrame([first]), pd.DataFrame(summaries)],\n        ignore_index=True,\n    )\n    combined.to_csv(\n        output_dir / "development_summary.csv", index=False\n    )\n    pd.DataFrame(metadata).to_csv(\n        output_dir / "training_metadata.csv", index=False\n    )\n\n    pooled = weighted_pooled(combined)\n    pd.DataFrame([{\n        "variant": preset_name,\n        **pooled,\n    }]).to_csv(output_dir / "pooled_summary.csv", index=False)\n\n    print(\n        f"\\n[{preset_name}] COMPLETE | "\n        f"pricing={pooled[\'pooled_otm_pricing_RMSE\']:.6f} | "\n        f"NLV hedge={pooled[\'pooled_otm_NLV_hedge_RMSE\']:.6f} | "\n        f"n={pooled[\'otm_n\']}",\n        flush=True,\n    )\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--preset",\n        required=True,\n        choices=sorted(PRESETS),\n    )\n    args = parser.parse_args()\n    run_variant(args.preset)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")

# Compile all support scripts before leaving anything running overnight.
for script in [WORK / "search_pipeline.py", WORK / "audit_v7.py", RUNNER]:
    subprocess.run([sys.executable, "-m", "py_compile", str(script)], check=True)

# Verify tqdm is installed and the runner imports.
import tqdm
print("tqdm:", tqdm.__version__)

test = subprocess.run(
    [sys.executable, str(RUNNER), "--help"],
    cwd=WORK,
    capture_output=True,
    text=True,
    check=True,
)
print(test.stdout.splitlines()[0])
print("Overnight runner syntax/import checks passed.")


tqdm: 4.66.5
usage: overnight_fixed_runner.py [-h] --preset
Overnight runner syntax/import checks passed.


## 1. Overnight queue

This single cell runs the two expensive branches **sequentially**:

`capped15_endpoint  →  capped20_twobatch`

The child runner shows:
- one progress bar across the four rolling calibration dates;
- one progress bar across the 15 or 20 SDE updates within the active date;
- live training-loss and hedge-variance-loss readouts.

If Python/Jupyter is interrupted, run the notebook again. The current date resumes from the most recent 2-update checkpoint and finished dates are skipped.


In [3]:
from tqdm.auto import tqdm

QUEUE = ["capped15_endpoint", "capped20_twobatch"]

for preset in tqdm(QUEUE, desc="Overnight experiment queue", unit="branch", dynamic_ncols=True):
    print("\n" + "=" * 78)
    print("STARTING:", preset)
    print("=" * 78)
    cmd = [
        sys.executable, "-u", "overnight_fixed_runner.py",
        "--preset", preset,
    ]
    result = subprocess.run(cmd, cwd=WORK)
    if result.returncode != 0:
        raise RuntimeError(
            f"{preset} exited with code {result.returncode}. "
            "Progress up to the latest 2-update checkpoint is preserved; "
            "rerun this notebook to resume."
        )
    print("FINISHED:", preset)

print("\nOvernight CPU queue complete.")


Overnight experiment queue:   0%|          | 0/2 [00:00<?, ?branch/s]


STARTING: capped15_endpoint
FINISHED: capped15_endpoint

STARTING: capped20_twobatch
FINISHED: capped20_twobatch

Overnight CPU queue complete.


## 2. Morning summary

This reports the pooled five-transition development metrics. The **same common Mar 1→4 transition** is prepended to every branch, so comparisons are like-for-like.

Reference values from the existing search:
- common-shock `final_safe` baseline pricing RMSE ≈ **4.4090**, NLV hedge RMSE ≈ **2.1051**
- `capped20_endpoint` pricing RMSE ≈ **4.2799**, NLV hedge RMSE ≈ **2.1093**
- high-patience `robust_mse` pricing RMSE ≈ **4.6236**, NLV hedge RMSE ≈ **2.1343**


In [4]:
import pandas as pd

rows = []
for name in ["capped15_endpoint", "capped20_twobatch"]:
    pooled = WORK / "search_runs" / name / "pooled_summary.csv"
    dev = WORK / "search_runs" / name / "development_summary.csv"

    if pooled.exists():
        frame = pd.read_csv(pooled)
        rows.append(frame.iloc[0].to_dict())
    elif dev.exists():
        frame = pd.read_csv(dev)
        n = frame["otm_n"].to_numpy(float)
        pricing = frame["otm_pricing_RMSE"].to_numpy(float)
        hedge = frame["otm_NLV_hedge_RMSE"].to_numpy(float)
        rows.append({
            "variant": name,
            "otm_n": int(n.sum()),
            "pooled_otm_pricing_RMSE":
                float((sum(n * pricing**2) / sum(n)) ** 0.5),
            "pooled_otm_NLV_hedge_RMSE":
                float((sum(n * hedge**2) / sum(n)) ** 0.5),
        })
    else:
        rows.append({"variant": name, "status": "not complete"})

summary = pd.DataFrame(rows)
display(summary)


,variant,otm_n,pooled_otm_pricing_RMSE,pooled_otm_NLV_hedge_RMSE,pooled_otm_market_delta_hedge_RMSE,weighted_otm_pricing_MAE
0,capped15_endpoint,20001,4.294568,2.103109,1.9004,3.468511
1,capped20_twobatch,20001,4.292801,2.116035,1.9004,3.374695


## 3. Package everything for analysis

Run this in the morning (or after an interruption if you want me to inspect partial progress). It creates `v7_overnight_cpu_results.zip`.

The ZIP includes both overnight branches, including histories, transition-level contract outputs, final checkpoints and any mid-date resume state that still exists.


In [5]:
OUT_ZIP = ROOT / "v7_overnight_cpu_results.zip"
include_names = [
    "capped15_endpoint",
    "capped20_twobatch",
]

with zipfile.ZipFile(
    OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for name in include_names:
        folder = WORK / "search_runs" / name
        if not folder.exists():
            continue
        for p in folder.rglob("*"):
            if p.is_file():
                z.write(p, arcname=str(p.relative_to(WORK)))
    z.write(
        WORK / "overnight_fixed_runner.py",
        arcname="overnight_fixed_runner.py",
    )

print("Created:", OUT_ZIP)
print("Size (MB):", round(OUT_ZIP.stat().st_size / 1024**2, 1))


Created: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\v7_overnight_cpu_results.zip
Size (MB): 11.6


### What not to queue tonight

Do **not** additionally run `robust_mse8k`, the old 100-update `robust_twobatch`, or the old long `robust_twostage`. The overnight two-batch run above already places the Sabaté-style estimator inside the empirically better **early-adaptation** regime.

After these results, the next CPU decision should be made from evidence rather than automatically launching more branches.
